# Fase 2 — Limpieza e ingeniería de datos (Flujo B, parte 1)

| Campo | Valor |
|---|---|
| **Rol líder** | Ingeniero de Datos |
| **Entrada** | `data/raw/chembl_panama_bioactivity.csv` |
| **Salida** | `data/processed/chembl_clean.csv` (sin NaN) |
| **Doc de fase** | [`docs/analisis_proyecto/fases/fase2_limpieza_datos.md`](../../docs/analisis_proyecto/fases/fase2_limpieza_datos.md) |

Secciones 0 y 2 del Flujo B: configuración, faltantes e imputación.


## 0. Configuración


In [ ]:
import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from IPython.display import Image, display
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    RocCurveDisplay,
    accuracy_score,
    classification_report,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.svm import SVC, SVR
from upsetplot import UpSet

ROOT = Path.cwd().parent.parent if Path.cwd().name == "proyecto analisis de datos" else (
    Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.analisis_proyecto.chembl_preprocessing import (
    ASSAY_FEATURE_COLS,
    FEATURE_COLS,
    NUMERIC_DESCRIPTOR_COLS,
    build_supervised_matrix,
    correlation_with_target,
    drop_columns_high_nan,
    encode_assay_features,
    evaluate_classification,
    evaluate_regression,
    get_available_feature_cols,
    get_feature_matrix,
    impute_median_by_family,
    load_bioactivity,
    missingness_upset_series,
    plot_missingno_report,
    numeric_and_categorical_cols,
    summary_statistics,
    train_test_split_by_group,
    train_test_split_rows,
)

sns.set_theme(style="whitegrid", palette="muted", font_scale=1.05)
plt.rcParams.update({"figure.figsize": (10, 5), "figure.dpi": 120})

RAW_CSV = ROOT / "data" / "raw" / "chembl_panama_bioactivity.csv"
CLEAN_CSV = ROOT / "data" / "processed" / "chembl_clean.csv"
FIG_DIR = ROOT / "outputs" / "chembl" / "figures"
MODEL_DIR = ROOT / "outputs" / "chembl" / "models"
RESULTS_DIR = ROOT / "outputs" / "chembl" / "results"

for d in (FIG_DIR, MODEL_DIR, RESULTS_DIR, CLEAN_CSV.parent):
    d.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42
NAN_COL_THRESHOLD = 250

assert RAW_CSV.exists(), (
    f"No se encontró {RAW_CSV}. Ejecuta primero fase1_adquisicion.ipynb"
)

df = load_bioactivity(RAW_CSV)
print(f"Dataset cargado: {df.shape[0]:,} filas × {df.shape[1]} columnas")
print(f"Compuestos: {df['compound_name'].nunique()}")


## 2. Valores faltantes e imputación

- Visualización con **missingno** y **UpSetPlot**
- Eliminar columnas con más de 250 NaN
- Imputar numéricas: **mediana por `family`** (fallback global)


In [ ]:
saved_msno = plot_missingno_report(df, FIG_DIR)
if saved_msno:
    for path in saved_msno:
        display(Image(filename=str(path)))
else:
    print("Sin valores faltantes — no se generan gráficos missingno.")


In [ ]:
upset_data, nan_cols = missingness_upset_series(df)
if upset_data is not None:
    # Varios registros comparten el mismo patrón de NaN → subset_size="count"
    upset = UpSet(upset_data, show_counts=True, sort_by="cardinality", subset_size="count")
    upset.plot()
    plt.suptitle("Patrones de valores faltantes (UpSetPlot)", y=1.02)
    plt.savefig(FIG_DIR / "upset_missing.png", bbox_inches="tight")
    plt.show()
else:
    print("Sin valores faltantes para UpSetPlot.")


In [ ]:
df_dropped, nan_report = drop_columns_high_nan(df, threshold=NAN_COL_THRESHOLD)
dropped_cols = nan_report.query("decision == 'eliminar'")["columna"].tolist()
print(f"Columnas eliminadas (>{NAN_COL_THRESHOLD} NaN): {dropped_cols}")
display(nan_report)

num_cols, cat_cols = numeric_and_categorical_cols(df_dropped)
df_clean = impute_median_by_family(df_dropped, numeric_cols=num_cols, categorical_cols=cat_cols)

remaining_nan = df_clean.isna().sum()
print("\nNaN restantes por columna (features de modelado):")
feat_nan = remaining_nan[get_available_feature_cols(df_clean) + ["activity_class", "pchembl_value"]]
display(feat_nan[feat_nan > 0].to_frame("n_nan") if (feat_nan > 0).any() else pd.DataFrame({"ok": ["Sin NaN en columnas clave"]}))

df_clean.to_csv(CLEAN_CSV, index=False)
print(f"\nGuardado: {CLEAN_CSV} ({len(df_clean):,} filas)")


---
*Fase anterior:* [`fase1_adquisicion.ipynb`](fase1_adquisicion.ipynb)  
*Siguiente fase:* [`fase3_eda.ipynb`](fase3_eda.ipynb)
